# High-z Example 06: Dynamical Mass vs Stellar Mass

**EPS Research — High-z Kinematic Corpus Z1**

For tier-1 rotators, dynamical mass Mdyn = V^2 * R / G.
Comparing Mdyn to M* reveals the dark matter fraction at z~5.
Two anomalous cases (CG32, DC396844) have Mdyn < M* —
physically implausible, likely due to inclination uncertainty.

**Corpus:** Flynn (2026), Zenodo DOI: 10.5281/zenodo.20369285  
**arXiv:** 2605.25339  
**Source:** Jones et al. (2021), MNRAS 507, 3540; Le Fevre et al. (2020)  
**Dependencies:** Python 3, numpy, matplotlib

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

with open('high_z_kinematic_corpus_Z1.json') as f:
    corpus = json.load(f)

galaxies  = corpus['galaxies']
rotators  = [g for g in galaxies if g.get('is_rotator') and g.get('quality_tier')==1]
print(f"Total galaxies: {len(galaxies)}")
print(f"Tier-1 rotators: {len(rotators)}")


In [ ]:
data = []
for g in rotators:
    last = g['data'][-1]
    log_mdyn = g.get('log_mdyn_msun')
    log_mstar = g.get('log_mstar_msun')
    if log_mdyn and log_mstar:
        data.append({'galaxy': g['galaxy'],
                     'z': g['redshift'],
                     'log_mdyn': log_mdyn,
                     'log_mstar': log_mstar,
                     'anomalous': log_mdyn < log_mstar})

print(f"Rotators with mass data: {len(data)}")
print(f"\n{'Galaxy':<20} {'log Mdyn':>9} {'log M*':>8} {'Mdyn/M*':>8} {'Flag'}")
print('-' * 58)
for d in sorted(data, key=lambda x: x['log_mdyn']):
    ratio = d['log_mdyn'] - d['log_mstar']
    flag  = '⚠️  Mdyn < M*' if d['anomalous'] else ''
    print(f"{d['galaxy']:<20} {d['log_mdyn']:>9.3f} {d['log_mstar']:>8.3f} "
          f"{ratio:>8.3f}  {flag}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
log_mdyn  = [d['log_mdyn']  for d in data]
log_mstar = [d['log_mstar'] for d in data]
colors    = ['#e74c3c' if d['anomalous'] else '#2166ac' for d in data]

ax.scatter(log_mstar, log_mdyn, s=80, c=colors, zorder=3, edgecolors='k', linewidths=0.5)
for d in data:
    ax.annotate(d['galaxy'][:8], (d['log_mstar'], d['log_mdyn']),
                textcoords='offset points', xytext=(5, 3), fontsize=7)

lim = [min(log_mstar+log_mdyn)-0.1, max(log_mstar+log_mdyn)+0.1]
ax.plot(lim, lim, 'k--', lw=1, alpha=0.4, label='Mdyn = M* (no dark matter)')
ax.set_xlabel(r'log M$_*$ ($M_\odot$)', fontsize=12)
ax.set_ylabel(r'log M$_{m dyn}$ ($M_\odot$)', fontsize=12)
ax.set_title('Dynamical vs Stellar Mass — Z1 Tier-1 Rotators\n'
             'Red = Mdyn < M* (anomalous, inclination uncertainty)',
             fontsize=10)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('hz06_mdyn_mstar.png', dpi=150, bbox_inches='tight')
plt.show()